# Rolling Time-Series Momentum on US Equity Indices
### A long/flat lookback study on the S&P 500 — in-sample optimisation, lookahead and overfitting controls, out-of-sample validation and a Nasdaq-100 (NQ) transfer test

*Quantitative research report · Data: Yahoo Finance via `yfinance` · Every number, chart and the verdict in §11 is computed when the notebook runs. Nothing is hard-coded.*

---

#### Abstract
We test one rule: **hold the index when its trailing return over a rolling calendar window is positive, otherwise hold T-bills.** Four lookbacks are evaluated: **1 week, 30 days, 90 days and 1 year**. The window rolls forward every trading day. On 27-Sep-2026, for example, the 1-year window compares against the close of 27-Sep-2025, or the last trading day before it.

The lookback is chosen **only from in-sample data (2000–2020)** using survival gates fixed before any results were seen. It is then frozen and evaluated **out of sample (2021–present)** against S&P 500 buy-and-hold. It is also transferred **without re-optimisation** to the Nasdaq-100.

The study contains:
* a lookahead-bias audit (truncation, future-perturbation and same-bar tests, plus a deliberately leaky backtest for contrast);
* an overfitting battery: a parameter-stability surface, the Deflated Sharpe Ratio, the Probability of Backtest Overfitting (via combinatorially symmetric cross-validation), walk-forward re-selection, a circular-shift timing test and an untouched 1990s holdout;
* robustness checks for transaction costs, execution delay and the cash assumption, plus a regime check on whether short-horizon momentum has decayed since the 1960s.

**Where this sits in the literature.** The rule is known as *time-series momentum* or *absolute momentum*. Key references are Moskowitz, Ooi & Pedersen (2012), *Time Series Momentum*, JFE; Hurst, Ooi & Pedersen (2017), *A Century of Evidence on Trend-Following Investing*; and Antonacci (2014), *Dual Momentum Investing*. Moving-average cousins appear in Faber (2007). Full references are in the Appendix.

#### Contents
1. Research question & pre-registered success criteria
2. Configuration: every research decision fixed up front
3. Data: acquisition, cleaning, quality control
4. Strategy specification & the backtests
5. Lookahead-bias audit
6. In-sample optimisation & survival gates
7. Overfitting diagnostics
8. Out-of-sample evaluation vs the S&P 500
9. Nasdaq-100 (NQ) transfer test
10. Robustness: costs, execution delay, cash, 1990s holdout, regime check, sub-periods
11. Conclusions (generated from the results)

Appendix: methodology notes, limitations, references

## 1. Research question & pre-registered success criteria

**Question.** Does the sign of the S&P 500's trailing return over a rolling window of length $L$ carry information about subsequent returns? If so, can a daily long/flat switch between the index and T-bills deliver better risk-adjusted returns than buy-and-hold?

* **H0 (no timing skill):** going to cash after a negative trailing return does not improve risk-adjusted performance. Any reduction in volatility only reflects lower average exposure.
* **H1:** the trailing-return sign predicts returns well enough that the long/flat strategy beats buy-and-hold on a risk-adjusted basis, net of costs, out of sample.

**What would count as success** (set before running the out-of-sample test):

| # | Criterion | Evidence |
|---|---|---|
| 1 | The selected lookback survives all in-sample gates | §6 |
| 2 | OOS Sharpe ≥ S&P 500 buy-and-hold, ideally with a significant bootstrap p-value | §8 |
| 3 | OOS maximum drawdown materially shallower than buy-and-hold | §8 |
| 4 | Low overfitting risk: PBO < 0.5, Deflated Sharpe > 0.95 | §7 |
| 5 | Timing skill beyond exposure (circular-shift test) | §8 |
| 6 | Transfers to the Nasdaq-100 with no re-fitting | §9 |
| 7 | Survives realistic costs and a one-day execution delay | §10 |

## 2. Configuration: every research decision fixed up front

| Decision | Choice | Rationale |
|---|---|---|
| Signal series | Price index (`^GSPC`, `^NDX`) | "Has the S&P gone up over the window?" is measured on the price chart a trader watches. |
| P&L series | Total-return index (`^SP500TR`, `^XNDX`) where available | Buy-and-hold must receive dividends. Otherwise the T-bill yield earned in cash gives the strategy an unfair edge. Long index futures plus T-bill collateral earn ≈ total return, so this also matches an ES/NQ futures implementation. |
| Cash | 13-week T-bill yield (`^IRX`), accrued from the **previous** close over elapsed calendar days | Earnable and known in advance. No lookahead. |
| In-sample (IS) | 2000-01-01 → 2020-12-31 (21 years) | Modern market structure (decimalisation, electronic trading, index futures and ETFs) that still contains the two major bear markets of 2000–02 and 2008, plus the 2020 crash. These episodes are what make or break a timing rule. Note: the window opens near the dot-com peak, which flatters timing relative to buy-and-hold early on. Gate G3 (positive Sharpe in *both* halves) guards against a result driven by that start date alone. |
| Out-of-sample (OOS) | 2021-01-01 → latest close (~5.8 years) | Never used for any choice. Contains the 2022 bear market (a slow grind lower, the kind trend rules handle well) and the 2025 tariff crash (a fast V-shaped reversal, the classic whipsaw). A Sharpe ratio over ~6 years carries a standard error of roughly ±0.4. |
| Pre-sample holdout | 1990-01-01 → 1999-12-31 | A second untouched decade *before* the in-sample window, with total-return data. It covers the 1990 recession, the 1998 LTCM crisis and the late-1990s bull market. |
| Execution | Signal at close *t* → position over *t+1* (`exec_lag = 1`) | A 2- and 3-day delay is tested in §10. |
| Costs | 5 bp per switch (one-way) | Conservative for ES/NQ futures or SPY/QQQ (≈1–2 bp). 0–50 bp is tested in §10. |
| Selection metric | IS Sharpe ratio among gate survivors | Gates are defined in §6. |

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from IPython.display import display, Markdown

try:
    import rolling_momentum as rm
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "rolling_momentum.py must be in the same folder as this notebook. Clone the whole repo: "
        "git clone https://github.com/tafall77/Rolling-momentum.git") from exc

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
rm.set_style()

CONFIG = {
    # data
    "data_dir": "data",            # CSV cache, created on first run
    "refresh_data": False,         # True -> re-download everything from Yahoo Finance
    "download_start": "1957-01-01",  # long history feeds the regime check in section 10.5
    # sample design (fixed before any result was inspected)
    "is_start": "2000-01-01",
    "is_end": "2020-12-31",
    "oos_start": "2021-01-01",
    "oos_end": None,               # None -> latest available close
    "presample_start": "1990-01-01",
    "presample_end": "1999-12-31",
    # implementation
    "cost_bps": 5.0,               # per switch (one-way): commission + slippage
    "exec_lag": 1,                 # signal at close t -> position over t+1
    # inference
    "n_boot": 2000,                # stationary-bootstrap replications
    "boot_block": 63,              # mean block length (trading days)
    "pbo_blocks": 16,              # CSCV blocks -> C(16, 8) = 12,870 splits
    "n_shift": 1000,               # circular-shift null draws
    "seed": 42,
}
TICKERS = {
    "S&P 500":    {"price": "^GSPC", "tr": "^SP500TR"},
    "Nasdaq-100": {"price": "^NDX",  "tr": "^XNDX"},
    "rf": "^IRX",                  # 13-week T-bill yield, % p.a.
}
CANDIDATES = rm.CANDIDATES         # {'1W', '30D', '90D', '1Y'} as calendar offsets
GRID_DAYS = [5, 7, 10, 14, 21, 30, 45, 60, 75, 90, 105, 120, 150, 180,
             210, 240, 270, 300, 330, 365, 400, 450, 500]
GRID = {f"{d}d": d for d in GRID_DAYS}
COST, LAG, SEED = CONFIG["cost_bps"], CONFIG["exec_lag"], CONFIG["seed"]
IS = (CONFIG["is_start"], CONFIG["is_end"])
IS_TXT = f"{IS[0][:4]}-{IS[1][:4]}"

PALETTE = {**rm.COLORS,
           "S&P 500 B&H": rm.COLORS["benchmark"],
           "Nasdaq-100 B&H": rm.COLORS["benchmark2"]}


def show(df, by="index"):
    display(rm.format_table(df, by=by))


def excess(frames, start, end):
    # Daily net excess returns (strategy minus cash) for a dict of backtests.
    return pd.DataFrame({k: rm.window(f["net"] - f["cash"], start, end) for k, f in frames.items()})


def sharpe_by(frames, start, end):
    return excess(frames, start, end).apply(rm.sharpe)


def lookback_days(lb):
    base = pd.Timestamp("2001-01-01")
    return (base + rm.as_offset(lb) - base).days

## 3. Data: acquisition, cleaning and quality control

Daily closes are downloaded from Yahoo Finance and cached in `data/`. Set `refresh_data=True` to update them. The download ends *before* today's date, so an unfinished intraday bar can never enter the sample.

| Series | Ticker | Role |
|---|---|---|
| S&P 500 price index | `^GSPC` | signal; P&L before total-return data exists |
| S&P 500 total return | `^SP500TR` | P&L (from 1988) |
| Nasdaq-100 price index | `^NDX` | signal; P&L fallback |
| Nasdaq-100 total return | `^XNDX` | P&L where available |
| 13-week T-bill yield | `^IRX` | cash leg and Sharpe-ratio hurdle |

In [ ]:
raw = {}
for mkt in ("S&P 500", "Nasdaq-100"):
    t = TICKERS[mkt]["price"]
    raw[t] = rm.load_close(t, CONFIG["download_start"], CONFIG["data_dir"], CONFIG["refresh_data"])
for t in (TICKERS["S&P 500"]["tr"], TICKERS["Nasdaq-100"]["tr"], TICKERS["rf"]):
    s = rm.load_close(t, CONFIG["download_start"], CONFIG["data_dir"], CONFIG["refresh_data"],
                      required=False)
    if s is not None:
        raw[t] = s

clean, quality = {}, {}
for t, s in raw.items():
    clean[t], quality[t] = rm.clean_close(s, positive=(t != TICKERS["rf"]))

print("Data-quality report (Yahoo Finance closes)")
display(pd.DataFrame(quality))

In [ ]:
rf = clean.get(TICKERS["rf"])
spx = rm.build_market("S&P 500", clean[TICKERS["S&P 500"]["price"]],
                      clean.get(TICKERS["S&P 500"]["tr"]), rf)
ndx = rm.build_market("Nasdaq-100", clean[TICKERS["Nasdaq-100"]["price"]],
                      clean.get(TICKERS["Nasdaq-100"]["tr"]), rf)


def describe_market(m):
    meta = m.meta
    if meta["tr_ticker"]:
        pnl = f"{meta['tr_ticker']} total return from {meta['tr_from'].date()} (price return before)"
    else:
        pnl = f"{meta['price_ticker']} price return (dividends excluded)"
    return {
        "signal series": meta["price_ticker"],
        "signal history from": m.close.index[0].date(),
        "P&L series": pnl,
        "cash series": (f"{meta['rf_ticker']} from {meta['rf_from'].date()}"
                        if meta["rf_ticker"] else "0% (T-bill series unavailable)"),
        "last close": m.close.index[-1].date(),
    }


display(pd.DataFrame({m.name: describe_market(m) for m in (spx, ndx)}))

# Guard rails: the sample design needs enough history before the in-sample start.
need = pd.Timestamp(CONFIG["is_start"]) - pd.DateOffset(days=max(GRID_DAYS))
for m in (spx, ndx):
    assert m.close.index[0] <= need, (
        f"{m.name}: history starts {m.close.index[0].date()}, the design needs {need.date()}")
    if m.meta["tr_from"] is None or m.meta["tr_from"] > pd.Timestamp(CONFIG["is_start"]):
        print(f"NOTE: {m.name} total-return data does not cover the whole in-sample period; "
              "price returns are used before it, which understates buy-and-hold by the dividend yield.")

OOS_END = spx.close.index[-1] if CONFIG["oos_end"] is None else min(pd.Timestamp(CONFIG["oos_end"]), spx.close.index[-1])
OOS_END = spx.close.index[spx.close.index <= OOS_END][-1]  # snap to the last trading day
OOS = (CONFIG["oos_start"], OOS_END)
OOS_TXT = f"{OOS[0][:4]}-{OOS_END.year}"
print(f"In-sample : {IS[0]} -> {IS[1]}")
print(f"Out-of-sample: {OOS[0]} -> {OOS_END.date()}")

In [ ]:
start = pd.Timestamp(CONFIG["is_start"])
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6.2), sharex=True,
                               gridspec_kw={"height_ratios": [3, 1.1]})
for m in (spx, ndx):
    w = (1 + m.ret.loc[start:]).cumprod() * 100
    ax1.plot(w.index, w.to_numpy(), color=PALETTE[f"{m.name} B&H"], lw=1.5, label=f"{m.name} (total return)")
    ax1.annotate(f"{w.iloc[-1]:,.0f}", (w.index[-1], w.iloc[-1]), xytext=(4, 0),
                 textcoords="offset points", fontsize=8, color=rm.INK["secondary"], va="center")
ax1.set_yscale("log")
ax1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax1.yaxis.set_minor_formatter(mtick.NullFormatter())
ax1.set_title(f"Index growth, {start.year} = 100 (log scale); shaded = out-of-sample")
ax1.legend(loc="upper left")
if rf is not None:
    r_ = rf.loc[start:]
    ax2.plot(r_.index, r_.to_numpy(), color=rm.INK["secondary"], lw=1.0)
ax2.set_title("13-week T-bill yield, % p.a. (cash leg)", fontsize=10)
for ax in (ax1, ax2):
    ax.axvspan(pd.Timestamp(CONFIG["oos_start"]), OOS_END, color="#f0efec", zorder=0, lw=0)
plt.tight_layout()
plt.show()

## 4. Strategy specification

For each trading day $t$ and calendar lookback $L \in \{1\text{W}, 30\text{D}, 90\text{D}, 1\text{Y}\}$:

$$m_t(L) = \frac{P_t}{P_{\tau(t-L)}} - 1, \qquad \tau(x) = \text{last trading day} \le x$$

$$s_t = \mathbb{1}\{m_t(L) > 0\}, \qquad w_{t+1} = s_t$$

$$r^{\text{strat}}_{t+1} = w_{t+1}\, r_{t+1} + (1 - w_{t+1})\, r^f_{t+1} - c\,\lvert w_{t+1} - w_t \rvert$$

* $P$ is the price index, $r$ the total return, $r^f$ the T-bill return and $c$ the cost per switch.
* The window is a **calendar** window that rolls every day, exactly as specified. When $t-L$ falls on a weekend or holiday, the last close before it is used. The table below shows this with live data.
* The signal is decided at the close and held over the *next* session. A zero return counts as "not positive" and means cash.
* The four lookbacks are approximately 5, 21, 63 and 252 trading days.

In [ ]:
def current_signals(m):
    last = m.close.index[-1]
    rows = []
    for name, lb in CANDIDATES.items():
        ref = rm.reference_date(m.close.index, last, lb)
        r = m.close[last] / m.close[ref] - 1
        rows.append({"lookback": name, "as of close": last.date(),
                     "window start (t - L)": (last - rm.as_offset(lb)).date(),
                     "reference close date": ref.date(),
                     "reference close": round(float(m.close[ref]), 2),
                     "latest close": round(float(m.close[last]), 2),
                     "trailing return": f"{r:+.2%}",
                     "position next session": "LONG" if r > 0 else "CASH"})
    return pd.DataFrame(rows).set_index("lookback")


for m in (spx, ndx):
    print(f"{m.name}: latest signals")
    display(current_signals(m))

ex = pd.Timestamp("2026-09-27")
print(f"Worked example from the brief: on {ex:%a %d-%b-%Y} the 1Y window starts "
      f"{ex - pd.DateOffset(years=1):%a %d-%b-%Y}; the as-of close used is the last trading day "
      f"on or before that date.")

### 4.1 Running the backtests
All backtests run once over each market's full history and are then *sliced* by date. The signal on any day depends only on data up to that day (verified in §5), so slicing after computing is equivalent to computing within each window. It also lets the 1-year signal be ready on day one of every sample.

In [ ]:
spx_frames = rm.run_lookbacks(spx, CANDIDATES, COST, LAG)
ndx_frames = rm.run_lookbacks(ndx, CANDIDATES, COST, LAG)
spx_grid = rm.run_lookbacks(spx, GRID, COST, LAG)       # parameter-stability grid (diagnostic)
ndx_grid = rm.run_lookbacks(ndx, GRID, COST, LAG)

exposure = pd.DataFrame({
    m: {k: rm.window(f["pos"], *IS).mean() for k, f in fr.items()}
    for m, fr in (("S&P 500", spx_frames), ("Nasdaq-100", ndx_frames))
})
switches = pd.DataFrame({
    m: {k: rm.window(f["turnover"], *IS).sum() / (len(rm.window(f, *IS)) / rm.TRADING_DAYS)
        for k, f in fr.items()}
    for m, fr in (("S&P 500", spx_frames), ("Nasdaq-100", ndx_frames))
})
print("In-sample time in market and switches per year")
display(pd.concat({"Time in market": exposure.map(lambda v: f"{v:.0%}"),
                   "Switches / year": switches.round(1)}, axis=1))

## 5. Lookahead-bias audit

A backtest has lookahead bias when a decision uses information that was not available when it was made. Four independent tests check this directly. A fifth exhibit shows what the bias would look like if it were present.

1. **Truncation invariance.** Signals computed on data that ends at a cutoff must match the full-sample signals up to that cutoff.
2. **Future perturbation.** Replace every price and return *after* a cutoff with random noise. Signals, positions and strategy returns up to the cutoff must not change.
3. **Same-bar test.** Shock only day *t*'s return. The position held over day *t* must not change, because a causal rule cannot react to the return it is about to earn. The test is run on the real rule (lag 1) and on a deliberately leaky version (lag 0) to prove it can detect the bias.
4. **Structural controls.** These are checked by construction and unit tests (`tests/`): the cash leg uses the previous close's yield; no partial bars are downloaded; the lookback is selected from in-sample rows only, which an assertion enforces.

In [ ]:
def snap(index, dates):
    pos = np.clip(index.searchsorted(dates), 0, len(index) - 1)
    return list(index[pos])


grid_dates = pd.date_range(CONFIG["is_start"], OOS_END, periods=12)
trunc = pd.concat({m.name: rm.truncation_check(m.close, CANDIDATES, snap(m.close.index, grid_dates))
                   for m in (spx, ndx)}, axis=1)
pert = pd.concat({m.name: rm.future_perturbation_check(m, CANDIDATES, snap(m.close.index, grid_dates[::3]),
                                                       COST, LAG, seed=SEED)
                  for m in (spx, ndx)}, axis=1)
print("5.1 Truncation invariance")
show(trunc, by="columns")
print("5.2 Future perturbation (signals, positions and net returns up to each cutoff)")
show(pert, by="columns")

In [ ]:
same_bar = pd.DataFrame({
    f"lag {LAG} (as traded): share changed": {k: rm.same_bar_check(spx, lb, lag=LAG, seed=SEED)
                                              for k, lb in CANDIDATES.items()},
    "lag 0 (leaky): share changed": {k: rm.same_bar_check(spx, lb, lag=0, seed=SEED)
                                     for k, lb in CANDIDATES.items()},
})
print("5.3 Same-bar test: share of 250 random days on which shocking that day's return "
      "changes the position held over it (S&P 500)")
show(same_bar, by="columns")

### 5.4 What lookahead bias would look like
Below is the same strategy computed **with** the classic off-by-one error: the position over day *t* is taken from the signal at the close of day *t*, which already contains day *t*'s return. Short lookbacks benefit most because one day is a large share of their window. The spectacular "leaky" numbers show why the lag matters, and why results that look too good to be true usually are.

In [ ]:
leaky_frames = rm.run_lookbacks(spx, CANDIDATES, COST, lag=0)
rows = {}
for k in CANDIDATES:
    for label, frames in (("as traded (lag 1)", spx_frames), ("LEAKY (lag 0)", leaky_frames)):
        f = rm.window(frames[k], *IS)
        rows[(k, label)] = rm.perf_stats(f["net"], f["cash"])[["CAGR", "Sharpe", "Max drawdown"]]
bh_is_stats = rm.perf_stats(rm.window(spx.ret, *IS), rm.window(spx.cash, *IS))
rows[("S&P 500 B&H", "")] = bh_is_stats[["CAGR", "Sharpe", "Max drawdown"]]
leak_tbl = pd.DataFrame(rows).T
print(f"In-sample {IS_TXT}, S&P 500")
show(leak_tbl, by="columns")

fig, ax = plt.subplots(figsize=(11, 4.2))
short = min(CANDIDATES, key=lambda k: lookback_days(CANDIDATES[k]))
rm.plot_wealth({f"Leaky {short} (lag 0)": rm.window(leaky_frames[short]["net"], *IS),
                f"{short} as traded": rm.window(spx_frames[short]["net"], *IS),
                "S&P 500 B&H": rm.window(spx.ret, *IS)},
               f"The cost of a one-day lookahead error ({short} lookback, in-sample)", ax=ax,
               colors={f"Leaky {short} (lag 0)": PALETTE["Leaky"], f"{short} as traded": PALETTE[short],
                       "S&P 500 B&H": PALETTE["S&P 500 B&H"]})
plt.tight_layout()
plt.show()

In [ ]:
def in_sample(frame):
    # The only gateway used for selection: slices to IS and asserts no OOS rows leak through.
    out = rm.window(frame, *IS)
    assert out.index.max() <= pd.Timestamp(CONFIG["is_end"]), "out-of-sample rows reached the selection step"
    return out


_ = [in_sample(f) for f in spx_frames.values()]   # exercised here, used throughout section 6

AUDIT = pd.DataFrame({
    "result": {
        "Truncation invariance (no dependence on future rows)": bool(trunc.all().all()),
        "Future perturbation leaves the past unchanged": bool(pert.all().all()),
        f"Same-bar test at lag {LAG}: position never reacts to the return it earns":
            bool((same_bar.iloc[:, 0] == 0).all()),
        "Same-bar test detects the deliberately leaky lag-0 version": bool((same_bar.iloc[:, 1] > 0).any()),
        "Cash leg accrues the previous close's yield (unit-tested)": True,
        "Partial current-day bar excluded (download end date is exclusive)": True,
        "Selection receives in-sample rows only (asserted)": True,
    }
})
AUDIT["result"] = AUDIT["result"].map({True: "PASS", False: "FAIL"})
print("Lookahead audit scorecard")
display(AUDIT)

## 6. In-sample optimisation & survival gates (2000–2020)

The four user-specified lookbacks compete **only on in-sample data**. A lookback "survives" if it passes all four gates, which were fixed before the results were seen:

| Gate | Requirement | Purpose |
|---|---|---|
| **G1** risk-adjusted | IS Sharpe ≥ buy-and-hold IS Sharpe | must add value per unit of risk |
| **G2** capital preservation | IS max drawdown shallower than buy-and-hold | the main reason to time an index |
| **G3** consistency | Sharpe > 0 in **both** halves of the IS period | rejects lookbacks that worked in one era only |
| **G4** cost robustness | still meets G1 at **2× costs** | rejects edges that exist only without friction |

Among survivors, the highest IS Sharpe is selected and **frozen**. If none survives, the best IS Sharpe is carried forward for documentation only and is flagged as such.

In [ ]:
is_frames = {k: in_sample(f) for k, f in spx_frames.items()}
is_series = {**{k: f["net"] for k, f in is_frames.items()}, "S&P 500 B&H": rm.window(spx.ret, *IS)}
is_table = rm.stats_table(is_series, rm.window(spx.cash, *IS), is_frames)
print("In-sample performance, S&P 500, net of 5 bp per switch")
show(is_table)

In [ ]:
def survival_gates(m, frames, frames_2x, start, end):
    s0, s1 = pd.Timestamp(start), pd.Timestamp(end)
    mid = s0 + (s1 - s0) / 2
    bh = rm.perf_stats(rm.window(m.ret, s0, s1), rm.window(m.cash, s0, s1))
    rows = {}
    for k, f in frames.items():
        w = rm.window(f, s0, s1)
        assert w.index.max() <= s1
        st = rm.perf_stats(w["net"], w["cash"])
        ex_ = w["net"] - w["cash"]
        h1, h2 = rm.sharpe(ex_.loc[:mid]), rm.sharpe(ex_.loc[mid + pd.Timedelta(days=1):])
        w2 = rm.window(frames_2x[k], s0, s1)
        sr2 = rm.sharpe(w2["net"] - w2["cash"])
        rows[k] = {
            "IS Sharpe": st["Sharpe"], "IS CAGR": st["CAGR"], "IS max DD": st["Max drawdown"],
            "Sharpe 1st half": h1, "Sharpe 2nd half": h2, "Sharpe @ 2x cost": sr2,
            "G1 Sharpe >= B&H": bool(st["Sharpe"] >= bh["Sharpe"]),
            "G2 MaxDD shallower": bool(st["Max drawdown"] > bh["Max drawdown"]),
            "G3 both halves > 0": bool(h1 > 0 and h2 > 0),
            "G4 G1 at 2x cost": bool(sr2 >= bh["Sharpe"]),
        }
    g = pd.DataFrame.from_dict(rows, orient="index")
    g["SURVIVES"] = g[[c for c in g.columns if c.startswith("G")]].all(axis=1)
    return g, bh


def pick(gates):
    surv = gates.index[gates["SURVIVES"]]
    if len(surv):
        return gates.loc[surv, "IS Sharpe"].astype(float).idxmax(), True
    return gates["IS Sharpe"].astype(float).idxmax(), False


spx_frames_2x = rm.run_lookbacks(spx, CANDIDATES, 2 * COST, LAG)
gates, bh_is = survival_gates(spx, spx_frames, spx_frames_2x, *IS)
SELECTED, SELECTED_SURVIVED = pick(gates)

print(f"Buy & hold (IS): Sharpe {bh_is['Sharpe']:.2f}, max drawdown {bh_is['Max drawdown']:.1%}")
show(gates, by="columns")
status = ("survived all four gates and has the highest in-sample Sharpe among survivors"
          if SELECTED_SURVIVED else
          "has the best in-sample Sharpe, but NO lookback survived every gate; it is carried "
          "forward for documentation only")
display(Markdown(f"### Selected lookback: **{SELECTED}**. It {status}. "
                 "It is now frozen for every out-of-sample and cross-market test."))

In [ ]:
crit = {"Sharpe": "Sharpe", "Sortino": "Sortino", "Calmar": "Calmar", "CAGR": "CAGR",
        "Max drawdown": "Max drawdown"}
agree = {name: is_table.loc[row, list(CANDIDATES)].astype(float).idxmax() for name, row in crit.items()}
print("Which lookback would each selection criterion pick in-sample?")
display(pd.Series(agree, name="best lookback").to_frame().T)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7.4), sharex=True, gridspec_kw={"height_ratios": [2.2, 1]})
series = {k: is_frames[k]["net"] for k in CANDIDATES} | {"S&P 500 B&H": rm.window(spx.ret, *IS)}
rm.plot_wealth(series, f"In-sample growth of $1 ({IS_TXT}, net of costs)", ax=ax1, colors=PALETTE)
rm.plot_drawdowns({SELECTED: is_frames[SELECTED]["net"], "S&P 500 B&H": rm.window(spx.ret, *IS)},
                  f"In-sample drawdowns: selected {SELECTED} vs buy & hold", ax=ax2, colors=PALETTE)
plt.tight_layout()
plt.show()

## 7. Overfitting diagnostics

Choosing the best of several backtests inflates the winner's apparent quality: the *winner's curse*. Five complementary tools measure how much of the in-sample result may be selection luck:

1. **Parameter-stability surface.** A robust edge sits on a broad plateau of neighbouring lookbacks, not on an isolated spike.
2. **Deflated Sharpe Ratio (DSR).** The probability that the true Sharpe is positive after accounting for the number of trials, their dispersion, and non-normal returns (Bailey & López de Prado, 2014).
3. **Probability of Backtest Overfitting (PBO) via CSCV.** The IS period is cut into 16 blocks. For all 12,870 half/half splits, we check whether the "in-sample" winner beats the median configuration on the other half (Bailey, Borwein, López de Prado & Zhu, 2017). PBO near 0 is good; 0.5 means selection is no better than a coin flip.
4. **Walk-forward re-selection.** The selection procedure itself is re-run every year using only past data. This tests whether *the process* of picking a lookback generalises, not just the one pick we made.
5. **IS → OOS degradation.** A post-hoc comparison of every candidate's in-sample and out-of-sample Sharpe.

In [ ]:
is_sr_grid, oos_sr_grid = sharpe_by(spx_grid, *IS), sharpe_by(spx_grid, *OOS)
bh_is_sr = rm.sharpe(rm.window(spx.ret - spx.cash, *IS))
bh_oos_sr = rm.sharpe(rm.window(spx.ret - spx.cash, *OOS))

fig, ax = plt.subplots(figsize=(11, 4.6))
ax.plot(GRID_DAYS, is_sr_grid.to_numpy(), color=rm.INK["primary"], marker="o", ms=4, lw=1.6,
        label=f"In-sample {IS_TXT} (used for selection)")
ax.plot(GRID_DAYS, oos_sr_grid.to_numpy(), color=rm.INK["muted"], marker="o", ms=4, lw=1.4,
        label="Out-of-sample (post-hoc diagnostic, never used to choose)")
ax.axhline(bh_is_sr, color=rm.INK["primary"], lw=0.8)
ax.axhline(bh_oos_sr, color=rm.INK["muted"], lw=0.8)
ax.annotate("buy & hold IS", (GRID_DAYS[-1], bh_is_sr), xytext=(4, 0), textcoords="offset points",
            fontsize=8, color=rm.INK["secondary"], va="center")
ax.annotate("buy & hold OOS", (GRID_DAYS[-1], bh_oos_sr), xytext=(4, 0), textcoords="offset points",
            fontsize=8, color=rm.INK["secondary"], va="center")
for name, lb in CANDIDATES.items():
    d = lookback_days(lb)
    v = rm.sharpe(in_sample(spx_frames[name])["net"] - in_sample(spx_frames[name])["cash"])
    ax.scatter([d], [v], s=80, color=PALETTE[name], zorder=5, edgecolor=rm.INK["surface"], linewidth=2)
    ax.annotate(name + (" (selected)" if name == SELECTED else ""), (d, v), xytext=(0, 11),
                textcoords="offset points", ha="center", fontsize=9, color=rm.INK["primary"], zorder=6,
                bbox=dict(boxstyle="round,pad=0.15", fc=rm.INK["surface"], ec="none"))
ax.set_xscale("log")
ax.set_xticks(GRID_DAYS[::2])
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v:g}"))
ax.xaxis.set_minor_formatter(mtick.NullFormatter())
ax.set_xlabel("Lookback (calendar days, log scale)")
ax.set_ylabel("Sharpe ratio (net)")
ax.set_title("7.1 Parameter stability: Sharpe ratio across 23 lookbacks, S&P 500")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

sel_days = lookback_days(CANDIDATES[SELECTED])
neigh = [k for k, d in GRID.items() if sel_days / 2 <= d <= sel_days * 2]
stability = pd.Series({
    f"IS Sharpe, selected ({SELECTED})": rm.sharpe(in_sample(spx_frames[SELECTED])["net"]
                                                   - in_sample(spx_frames[SELECTED])["cash"]),
    "IS Sharpe, median of neighbours (0.5x-2x lookback)": is_sr_grid[neigh].median(),
    "Share of all grid lookbacks beating B&H IS Sharpe": (is_sr_grid >= bh_is_sr).mean(),
    "Share of neighbours beating B&H IS Sharpe": (is_sr_grid[neigh] >= bh_is_sr).mean(),
    "Spearman rank corr., IS vs OOS Sharpe across grid": is_sr_grid.corr(oos_sr_grid, method="spearman"),
}, name="value")
show(stability.to_frame())

In [ ]:
sel_ex_is = in_sample(spx_frames[SELECTED])["net"] - in_sample(spx_frames[SELECTED])["cash"]
cand_sr_daily = excess(spx_frames, *IS).apply(lambda x: rm.sharpe(x, annualize=False)).to_numpy()
grid_sr_daily = excess(spx_grid, *IS).apply(lambda x: rm.sharpe(x, annualize=False)).to_numpy()
dsr = pd.DataFrame({
    "Candidate set (N=4)": rm.deflated_sharpe(sel_ex_is, cand_sr_daily),
    f"Full grid (N={len(GRID_DAYS)})": rm.deflated_sharpe(sel_ex_is, grid_sr_daily),
})
psr_vs_bh = rm.probabilistic_sharpe(sel_ex_is, bh_is_sr / np.sqrt(rm.TRADING_DAYS))
print(f"7.2 Sharpe-ratio inference for the selected lookback ({SELECTED}), in-sample")
show(dsr)
print(f"PSR against buy & hold's observed IS Sharpe ({bh_is_sr:.2f}): {psr_vs_bh:.1%}")

In [ ]:
pbo_cand = rm.pbo_cscv(excess(spx_frames, *IS), CONFIG["pbo_blocks"])
pbo_grid = rm.pbo_cscv(excess(spx_grid, *IS), CONFIG["pbo_blocks"])
pbo_tbl = pd.DataFrame({
    name: {"configs": float(p["configs"]), "splits": float(p["splits"]), "PBO": p["pbo"],
           "Prob. IS-best loses money OOS": p["prob_oos_loss"],
           "Degradation slope (OOS SR on IS SR)": p["degradation_slope"]}
    for name, p in (("Candidate set", pbo_cand), ("Full grid", pbo_grid))
})
print("7.3 Probability of Backtest Overfitting (CSCV, in-sample data only)")
show(pbo_tbl)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, (name, p) in zip(axes[:2], (("Candidate set (N=4)", pbo_cand), (f"Full grid (N={len(GRID_DAYS)})", pbo_grid))):
    ax.hist(p["logits"], bins=30, color=rm.INK["secondary"], edgecolor=rm.INK["surface"], linewidth=0.8)
    ax.axvline(0, color=PALETTE["Leaky"], lw=1.2)
    ax.set_title(f"{name}: PBO = {p['pbo']:.0%}", fontsize=10)
    ax.set_xlabel("logit of IS-winner's OOS rank (left of 0 = overfit)")
axes[2].scatter(pbo_grid["is_sharpe"], pbo_grid["oos_sharpe"], s=6, alpha=0.25, color=rm.INK["secondary"])
xs = np.linspace(pbo_grid["is_sharpe"].min(), pbo_grid["is_sharpe"].max(), 10)
axes[2].plot(xs, pbo_grid["degradation_intercept"] + pbo_grid["degradation_slope"] * xs,
             color=rm.INK["primary"], lw=1.4)
axes[2].set_title("Grid: IS-winner Sharpe, IS half vs OOS half", fontsize=10)
axes[2].set_xlabel("Sharpe in selection half")
axes[2].set_ylabel("Sharpe in held-out half")
plt.tight_layout()
plt.show()

In [ ]:
wf_bt, wf_log = rm.walk_forward(spx_frames, start=CONFIG["is_start"], min_train_years=5, cost_bps=COST)
wf_counts = wf_log["chosen"].value_counts().reindex(list(CANDIDATES)).fillna(0).astype(int)
oos_choices = wf_log[pd.to_datetime(wf_log["refit date"]) >= pd.Timestamp(CONFIG["is_end"])]["chosen"]
print("7.4 Walk-forward (expanding window, annual refit): how often each lookback was chosen")
display(pd.DataFrame({"all refits": wf_counts,
                      "refits for OOS years": oos_choices.value_counts().reindex(list(CANDIDATES)).fillna(0).astype(int)}).T)

fig, ax = plt.subplots(figsize=(11, 2.6))
order = list(CANDIDATES)
yrs = pd.to_datetime(wf_log["refit date"]).dt.year + 1
for name in order:
    mask = wf_log["chosen"] == name
    ax.scatter(yrs[mask], [order.index(name)] * mask.sum(), color=PALETTE[name], s=60,
               edgecolor=rm.INK["surface"], linewidth=2, label=name, zorder=3)
ax.set_yticks(range(len(order)))
ax.set_yticklabels(order)
ax.axvline(pd.Timestamp(CONFIG["oos_start"]).year - 0.5, color=rm.INK["muted"], lw=1)
ax.set_title("Lookback chosen for each year by walk-forward re-selection (vertical line = OOS start)")
ax.set_ylim(-0.6, len(order) - 0.4)
plt.tight_layout()
plt.show()

wf_oos = rm.window(wf_bt, *OOS)
wf_tbl = rm.stats_table({"Walk-forward": wf_oos["net"],
                         f"Fixed {SELECTED} (frozen {IS[1][:4]})": rm.window(spx_frames[SELECTED]["net"], *OOS),
                         "S&P 500 B&H": rm.window(spx.ret, *OOS)},
                        rm.window(spx.cash, *OOS),
                        {"Walk-forward": wf_oos,
                         f"Fixed {SELECTED} (frozen {IS[1][:4]})": rm.window(spx_frames[SELECTED], *OOS)})
print("Out-of-sample comparison: walk-forward selection vs the frozen choice")
show(wf_tbl.loc[["CAGR", "Ann. volatility", "Sharpe", "Max drawdown", "Calmar", "Time in market", "Switches / year"]])

In [ ]:
deg = pd.DataFrame({
    "IS Sharpe": sharpe_by(spx_frames, *IS),
    "OOS Sharpe": sharpe_by(spx_frames, *OOS),
})
deg["IS rank"] = deg["IS Sharpe"].rank(ascending=False).astype(int)
deg["OOS rank"] = deg["OOS Sharpe"].rank(ascending=False).astype(int)
deg["Change"] = deg["OOS Sharpe"] - deg["IS Sharpe"]
deg.loc["S&P 500 B&H"] = [bh_is_sr, bh_oos_sr, np.nan, np.nan, bh_oos_sr - bh_is_sr]
print("7.5 IS -> OOS degradation (post-hoc: computed after the choice was frozen)")
display(deg.style.format({"IS Sharpe": "{:.2f}", "OOS Sharpe": "{:.2f}", "Change": "{:+.2f}",
                          "IS rank": "{:.0f}", "OOS rank": "{:.0f}"}, na_rep="-"))

## 8. Out-of-sample evaluation vs the S&P 500 (2021 → present)

The selected lookback was frozen with data ending 31-Dec-2020. Everything in this section is a genuine out-of-sample test of that one decision. The other lookbacks are shown for context and are labelled post-hoc; they were not decisions anyone could have made at the end of 2020.

**Sample size.** About six years of out-of-sample data gives a Sharpe-ratio standard error of roughly ±0.4. That is enough to see whether the rule behaved as designed through the 2022 bear market and the 2025 crash, but modest differences from buy-and-hold will not be statistically significant.

In [ ]:
LBL = f"{SELECTED} momentum"
sel_oos = rm.window(spx_frames[SELECTED], *OOS)
bh_oos = rm.window(spx.ret, *OOS)
cash_oos = rm.window(spx.cash, *OOS)
others = {f"{k} (post-hoc)": rm.window(f, *OOS) for k, f in spx_frames.items() if k != SELECTED}
oos_table = rm.stats_table({LBL: sel_oos["net"], "S&P 500 B&H": bh_oos,
                            **{k: f["net"] for k, f in others.items()}},
                           cash_oos, {LBL: sel_oos, **others})
print(f"Out-of-sample performance, {OOS[0]} -> {OOS_END.date()}, net of {COST:g} bp per switch")
show(oos_table)

In [ ]:
colors = {LBL: PALETTE[SELECTED], "S&P 500 B&H": PALETTE["S&P 500 B&H"]}
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(11, 9.2), sharex=True,
                                    gridspec_kw={"height_ratios": [2.4, 0.5, 1.3]})
rm.plot_wealth({LBL: sel_oos["net"], "S&P 500 B&H": bh_oos},
               "Out-of-sample growth of $1 vs S&P 500 buy & hold", ax=ax1, colors=colors)
ax2.fill_between(sel_oos.index, sel_oos["pos"].fillna(0).to_numpy(), step="post",
                 color=PALETTE[SELECTED], alpha=0.35, lw=0)
ax2.set_yticks([0, 1])
ax2.set_yticklabels(["cash", "long"])
ax2.set_title("Position", fontsize=10)
ax2.grid(False)
rm.plot_drawdowns({LBL: sel_oos["net"], "S&P 500 B&H": bh_oos}, "Drawdowns", ax=ax3, colors=colors)
plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={"width_ratios": [1.15, 1]})
ROLL = 252 if len(sel_oos) > 2 * 252 else 63      # 12-month window needs a long OOS; else 3-month
roll = pd.DataFrame({LBL: (1 + sel_oos["net"]).rolling(ROLL).apply(np.prod, raw=True) - 1,
                     "S&P 500 B&H": (1 + bh_oos).rolling(ROLL).apply(np.prod, raw=True) - 1})
for c in roll:
    ax1.plot(roll.index, roll[c].to_numpy(), color=colors[c], lw=1.4, label=c,
             zorder=2 if "B&H" in c else 3)
ax1.axhline(0, color=rm.INK["axis"], lw=0.8)
ax1.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
ax1.set_title(f"Rolling {'12' if ROLL == 252 else '3'}-month return")
ax1.legend(loc="lower left")

ann = pd.DataFrame({LBL: rm.annual_returns(sel_oos["net"]), "S&P 500 B&H": rm.annual_returns(bh_oos)})
x = np.arange(len(ann))
wdt = 0.4
for i, c in enumerate(ann):
    ax2.bar(x + (i - 0.5) * wdt, ann[c].to_numpy(), width=wdt - 0.03, color=colors[c], label=c)
ax2.set_xticks(x)
ax2.set_xticklabels([str(y)[2:] for y in ann.index])
ax2.axhline(0, color=rm.INK["axis"], lw=0.8)
ax2.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
ax2.set_title("Calendar-year returns ('YY; final year is partial)")
ax2.legend(loc="best")
plt.tight_layout()
plt.show()

ann_tbl = ann.copy()
ann_tbl["Difference"] = ann_tbl[LBL] - ann_tbl["S&P 500 B&H"]
display(ann_tbl.T.style.format("{:+.1%}"))

### 8.1 Statistical tests (out-of-sample)
* **Sharpe-difference bootstrap.** A paired stationary bootstrap (Politis & Romano, 1994; mean block of 63 days) of SR(strategy) − SR(buy-and-hold). It preserves autocorrelation and the volatility clustering shared by both series.
* **CAPM alpha.** The regression of strategy excess returns on S&P excess returns, with Newey–West (HAC) standard errors.
* **Circular-shift timing test.** The position series is rotated by random offsets. This keeps exposure, switch frequency and spell lengths identical, but breaks the link to future returns. If the real strategy cannot beat its rotations, its edge is only lower average exposure, not timing.

In [ ]:
boot = rm.bootstrap_sharpe_difference(sel_oos["net"] - cash_oos, bh_oos - cash_oos,
                                      CONFIG["n_boot"], CONFIG["boot_block"], SEED)
ab = rm.alpha_beta(sel_oos["net"] - cash_oos, bh_oos - cash_oos)
shift = rm.circular_shift_test(sel_oos, COST, n=CONFIG["n_shift"], seed=SEED)
cap = rm.capture_ratios(sel_oos["net"], bh_oos)
tests_tbl = pd.concat([
    pd.Series({k: v for k, v in boot.items() if k != "draws"}),
    ab,
    pd.Series({"Circular-shift Sharpe (actual)": shift["actual"],
               "Circular-shift null 95th pct Sharpe": shift["null 95th pct"],
               "Circular-shift p-value": shift["p-value"]}),
    cap,
])
show(tests_tbl.to_frame(f"{LBL} vs S&P 500 B&H (OOS)"))
print("Invested-spell statistics (OOS)")
show(rm.trade_stats(sel_oos).to_frame(LBL))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 3.6))
ax1.hist(boot["draws"], bins=50, color=rm.INK["secondary"], edgecolor=rm.INK["surface"], linewidth=0.6)
ax1.axvline(0, color=rm.INK["axis"], lw=1)
ax1.axvline(boot["SR difference"], color=PALETTE[SELECTED], lw=2)
ax1.set_title(f"Bootstrap SR(strategy) - SR(B&H): observed {boot['SR difference']:+.2f}, "
              f"95% CI [{boot['95% CI low']:+.2f}, {boot['95% CI high']:+.2f}]", fontsize=10)
ax2.hist(shift["null"], bins=50, color=rm.INK["secondary"], edgecolor=rm.INK["surface"], linewidth=0.6)
ax2.axvline(shift["actual"], color=PALETTE[SELECTED], lw=2)
ax2.set_title(f"Circular-shift null vs actual Sharpe: p = {shift['p-value']:.3f}", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
CRISES = [
    ("Gulf War sell-off", "1990-07-16", "1990-10-11"),
    ("LTCM / Russia", "1998-07-17", "1998-10-08"),
    ("Dot-com bust", "2000-03-24", "2002-10-09"),
    ("Global Financial Crisis", "2007-10-09", "2009-03-09"),
    ("US downgrade 2011", "2011-04-29", "2011-10-03"),
    ("China / oil 2015-16", "2015-05-21", "2016-02-11"),
    ("Q4 2018", "2018-09-20", "2018-12-24"),
    ("COVID crash", "2020-02-19", "2020-03-23"),
    ("2022 bear market", "2022-01-03", "2022-10-12"),
    ("2025 tariff shock", "2025-02-19", "2025-04-08"),
]


def period_ret(r, a, b):
    w = r.loc[pd.Timestamp(a) + pd.Timedelta(days=1):pd.Timestamp(b)]
    return float((1 + w).prod() - 1) if len(w) else np.nan


rows = []
for name, a, b in CRISES:
    if pd.Timestamp(a) < pd.Timestamp(CONFIG["presample_start"]) or pd.Timestamp(b) > OOS_END:
        continue
    rebound_end = pd.Timestamp(b) + pd.DateOffset(months=3)
    rows.append({
        "episode": name, "peak": a, "trough": b,
        "sample": ("OOS" if pd.Timestamp(a) >= pd.Timestamp(CONFIG["oos_start"])
                   else "IS" if pd.Timestamp(a) >= pd.Timestamp(CONFIG["is_start"]) else "holdout"),
        "S&P B&H": period_ret(spx.ret, a, b),
        f"S&P {SELECTED}": period_ret(spx_frames[SELECTED]["net"], a, b),
        "NQ B&H": period_ret(ndx.ret, a, b),
        f"NQ {SELECTED}": period_ret(ndx_frames[SELECTED]["net"], a, b),
        "S&P B&H next 3m": period_ret(spx.ret, b, rebound_end),
        f"S&P {SELECTED} next 3m": period_ret(spx_frames[SELECTED]["net"], b, rebound_end),
    })
crisis = pd.DataFrame(rows).set_index("episode")
print("Peak-to-trough episodes (S&P 500 dates) and the 3 months after each trough")
pct_cols = [c for c in crisis.columns if c not in ("peak", "trough", "sample")]
display(crisis.style.format({c: "{:+.1%}" for c in pct_cols}, na_rep="-"))

## 9. Nasdaq-100 (NQ) transfer test

The lookback frozen on the S&P 500 is applied to the Nasdaq-100 **with no re-fitting**. No Nasdaq data was used for any decision, so the *entire* Nasdaq history is out-of-market evidence, and 2021 onward is also out-of-time. We also run the Nasdaq through the same survival gates on its own in-sample period to see whether it would independently choose the same lookback.

In [ ]:
NQ_LBL = f"{SELECTED} momentum on NQ"
nq_cash_oos = rm.window(ndx.cash, *OOS)
nq_sel_oos = rm.window(ndx_frames[SELECTED], *OOS)
nq_oos_table = rm.stats_table({NQ_LBL: nq_sel_oos["net"], "Nasdaq-100 B&H": rm.window(ndx.ret, *OOS),
                               "S&P 500 B&H": bh_oos},
                              nq_cash_oos, {NQ_LBL: nq_sel_oos})
FULL = (CONFIG["is_start"], OOS_END)
nq_sel_full = rm.window(ndx_frames[SELECTED], *FULL)
nq_full_table = rm.stats_table({NQ_LBL: nq_sel_full["net"], "Nasdaq-100 B&H": rm.window(ndx.ret, *FULL),
                                "S&P 500 B&H": rm.window(spx.ret, *FULL)},
                               rm.window(ndx.cash, *FULL), {NQ_LBL: nq_sel_full})
keep = ["CAGR", "Ann. volatility", "Sharpe", "Sortino", "Max drawdown", "Calmar",
        "Longest drawdown (yrs)", "Time in market", "Switches / year"]
print(f"Out-of-sample {OOS[0]} -> {OOS_END.date()}")
show(nq_oos_table.loc[keep])
print(f"Full history {FULL[0]} -> {OOS_END.date()} (entirely out-of-market)")
show(nq_full_table.loc[keep])

In [ ]:
nq_colors = {NQ_LBL: PALETTE[SELECTED], "Nasdaq-100 B&H": PALETTE["Nasdaq-100 B&H"],
             "S&P 500 B&H": PALETTE["S&P 500 B&H"]}
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7.4), sharex=True, gridspec_kw={"height_ratios": [2.2, 1]})
rm.plot_wealth({NQ_LBL: nq_sel_full["net"], "Nasdaq-100 B&H": rm.window(ndx.ret, *FULL),
                "S&P 500 B&H": rm.window(spx.ret, *FULL)},
               f"Nasdaq-100: growth of $1 since {FULL[0][:4]} (shaded = out-of-sample)", ax=ax1, colors=nq_colors)
rm.plot_drawdowns({NQ_LBL: nq_sel_full["net"], "Nasdaq-100 B&H": rm.window(ndx.ret, *FULL)},
                  "Nasdaq-100 drawdowns", ax=ax2, colors=nq_colors)
for ax in (ax1, ax2):
    ax.axvspan(pd.Timestamp(CONFIG["oos_start"]), OOS_END, color="#f0efec", zorder=0, lw=0)
plt.tight_layout()
plt.show()

In [ ]:
nq_boot_oos = rm.bootstrap_sharpe_difference(nq_sel_oos["net"] - nq_cash_oos,
                                             rm.window(ndx.ret, *OOS) - nq_cash_oos,
                                             CONFIG["n_boot"], CONFIG["boot_block"], SEED)
nq_boot_full = rm.bootstrap_sharpe_difference(nq_sel_full["net"] - rm.window(ndx.cash, *FULL),
                                              rm.window(ndx.ret - ndx.cash, *FULL),
                                              CONFIG["n_boot"], CONFIG["boot_block"], SEED)
nq_shift = rm.circular_shift_test(nq_sel_oos, COST, n=CONFIG["n_shift"], seed=SEED)
nq_tests = pd.DataFrame({
    "OOS": {**{k: v for k, v in nq_boot_oos.items() if k != "draws"},
            "Circular-shift p-value": nq_shift["p-value"]},
    "Full history": {k: v for k, v in nq_boot_full.items() if k != "draws"},
})
print(f"{NQ_LBL} vs Nasdaq-100 buy & hold")
show(nq_tests)

ndx_frames_2x = rm.run_lookbacks(ndx, CANDIDATES, 2 * COST, LAG)
nq_gates, nq_bh_is = survival_gates(ndx, ndx_frames, ndx_frames_2x, *IS)
NQ_OWN, NQ_OWN_SURVIVED = pick(nq_gates)
print(f"Nasdaq-100's own in-sample gates (B&H IS Sharpe {nq_bh_is['Sharpe']:.2f}, "
      f"max DD {nq_bh_is['Max drawdown']:.1%})")
show(nq_gates, by="columns")
same = "the SAME lookback" if NQ_OWN == SELECTED else f"a different lookback ({NQ_OWN})"
display(Markdown(f"Run independently, the Nasdaq-100 in-sample data would choose **{same}** "
                 f"(S&P choice: **{SELECTED}**)."))

In [ ]:
nq_is_grid, nq_oos_grid = sharpe_by(ndx_grid, *IS), sharpe_by(ndx_grid, *OOS)
nq_bh_is_sr = rm.sharpe(rm.window(ndx.ret - ndx.cash, *IS))
nq_bh_oos_sr = rm.sharpe(rm.window(ndx.ret - ndx.cash, *OOS))
fig, ax = plt.subplots(figsize=(11, 4.2))
ax.plot(GRID_DAYS, nq_is_grid.to_numpy(), color=rm.INK["primary"], marker="o", ms=4, lw=1.6, label=f"In-sample {IS_TXT}")
ax.plot(GRID_DAYS, nq_oos_grid.to_numpy(), color=rm.INK["muted"], marker="o", ms=4, lw=1.4, label="Out-of-sample")
ax.axhline(nq_bh_is_sr, color=rm.INK["primary"], lw=0.8)
ax.axhline(nq_bh_oos_sr, color=rm.INK["muted"], lw=0.8)
ax.annotate("B&H IS", (GRID_DAYS[-1], nq_bh_is_sr), xytext=(4, 0), textcoords="offset points", fontsize=8, va="center")
ax.annotate("B&H OOS", (GRID_DAYS[-1], nq_bh_oos_sr), xytext=(4, 0), textcoords="offset points", fontsize=8, va="center")
d = lookback_days(CANDIDATES[SELECTED])
ax.axvline(d, color=PALETTE[SELECTED], lw=1.2)
ax.annotate(f"{SELECTED} (frozen on S&P)", (d, ax.get_ylim()[1]), xytext=(4, -12), textcoords="offset points", fontsize=9)
ax.set_xscale("log")
ax.set_xticks(GRID_DAYS[::2])
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v:g}"))
ax.xaxis.set_minor_formatter(mtick.NullFormatter())
ax.set_xlabel("Lookback (calendar days, log scale)")
ax.set_ylabel("Sharpe ratio (net)")
ax.set_title("Nasdaq-100 parameter stability: Sharpe ratio across 23 lookbacks")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 10. Robustness

### 10.1 Transaction costs
Out-of-sample Sharpe for each lookback across per-switch costs from 0 to 50 bp. The **break-even cost** is the per-switch cost at which a lookback's OOS Sharpe falls to buy-and-hold's.

In [ ]:
COST_GRID = [0, 1, 2, 5, 10, 20, 50]
rows = []
for c in COST_GRID:
    for k, f in rm.run_lookbacks(spx, CANDIDATES, c, LAG).items():
        w = rm.window(f, *OOS)
        rows.append({"cost (bp)": c, "lookback": k, "Sharpe": rm.sharpe(w["net"] - w["cash"]),
                     "CAGR": rm.perf_stats(w["net"], w["cash"])["CAGR"]})
cost_df = pd.DataFrame(rows)
cost_sr = cost_df.pivot(index="cost (bp)", columns="lookback", values="Sharpe")[list(CANDIDATES)]
cost_cagr = cost_df.pivot(index="cost (bp)", columns="lookback", values="CAGR")[list(CANDIDATES)]


def breakeven_cost(frame, target_sr, hi=500.0):
    # Per-switch cost (bp) at which OOS Sharpe equals target_sr (bisection; Sharpe falls with cost).
    w = rm.window(frame, *OOS)
    if w["turnover"].sum() == 0:
        return None                      # no trades out of sample: cost is irrelevant
    pos, ret, cash = w["pos"].fillna(0).to_numpy(), w["asset"].to_numpy(), w["cash"].to_numpy()
    f = lambda c: rm.sharpe(rm.positions_to_returns(pos, ret, cash, c) - cash) - target_sr
    if f(0.0) < 0:
        return np.nan
    if f(hi) > 0:
        return np.inf
    lo_, hi_ = 0.0, hi
    for _ in range(60):
        mid = (lo_ + hi_) / 2
        lo_, hi_ = (mid, hi_) if f(mid) > 0 else (lo_, mid)
    return (lo_ + hi_) / 2


be = pd.Series({k: breakeven_cost(spx_frames[k], bh_oos_sr) for k in CANDIDATES},
               name="break-even cost vs B&H (bp per switch)", dtype=object)


def fmt_be(v):
    if v is None:
        return "no OOS trades"
    return "never beats B&H" if np.isnan(v) else ("> 500" if np.isinf(v) else f"{v:.1f}")

print("OOS Sharpe by cost (rows: bp per switch)")
display(cost_sr.style.format("{:.2f}"))
print("OOS CAGR by cost")
display(cost_cagr.style.format("{:.1%}"))
display(be.map(fmt_be).to_frame().T)

fig, ax = plt.subplots(figsize=(11, 4))
for k in CANDIDATES:
    ax.plot(COST_GRID, cost_sr[k].to_numpy(), color=PALETTE[k], marker="o", ms=4, label=k)
ax.axhline(bh_oos_sr, color=PALETTE["S&P 500 B&H"], lw=1)
ax.annotate("S&P 500 B&H", (COST_GRID[-1], bh_oos_sr), xytext=(4, 0), textcoords="offset points", fontsize=8, va="center")
ax.set_xlabel("Cost per switch (bp)")
ax.set_ylabel("OOS Sharpe")
ax.set_title("Out-of-sample Sharpe vs transaction cost, S&P 500")
ax.legend(loc="lower left", ncol=4)
plt.tight_layout()
plt.show()

### 10.2 Execution delay and 10.3 cash assumption
* **Delay.** Acting one or two sessions *later* than the signal (lag 2, 3) shows how much the result depends on precise timing.
* **Cash.** Earning 0% instead of T-bills while flat, still judged against the T-bill hurdle, removes the carry earned in cash from the result.

In [ ]:
rows = {}
for m in (spx, ndx):
    for lag in (1, 2, 3):
        f = rm.backtest(rm.signal(m.close, CANDIDATES[SELECTED]), m.ret, m.cash, COST, lag)
        for per, (a, b) in (("IS", IS), ("OOS", OOS)):
            w = rm.window(f, a, b)
            st = rm.perf_stats(w["net"], w["cash"])
            rows[(m.name, f"lag {lag}" + (" (base)" if lag == LAG else ""), per)] = st[["CAGR", "Sharpe", "Max drawdown"]]
lag_tbl = pd.DataFrame(rows).T.unstack(level=2)
lag_tbl.columns = [f"{a} {b}" for a, b in lag_tbl.columns]
lag_tbl = lag_tbl.reindex(["S&P 500", "Nasdaq-100"], level=0)
print(f"10.2 Execution delay, {SELECTED} lookback")
show(lag_tbl, by="columns")

spx0 = rm.build_market("S&P 500", spx.close, clean.get(TICKERS["S&P 500"]["tr"]), None)
ndx0 = rm.build_market("Nasdaq-100", ndx.close, clean.get(TICKERS["Nasdaq-100"]["tr"]), None)
rows = {}
for m, m0 in ((spx, spx0), (ndx, ndx0)):
    for label, mm in (("T-bills (base)", m), ("0% cash", m0)):
        f = rm.backtest(rm.signal(mm.close, CANDIDATES[SELECTED]), mm.ret, mm.cash, COST, LAG)
        w = rm.window(f["net"], *OOS)
        rows[(m.name, label)] = rm.perf_stats(w, rm.window(m.cash, *OOS))[["CAGR", "Sharpe", "Max drawdown"]]
    rows[(m.name, "buy & hold")] = rm.perf_stats(rm.window(m.ret, *OOS), rm.window(m.cash, *OOS))[["CAGR", "Sharpe", "Max drawdown"]]
print(f"10.3 Cash assumption, {SELECTED} lookback, OOS (Sharpe always vs T-bills)")
show(pd.DataFrame(rows).T, by="columns")

### 10.4 Pre-sample holdout (1990–1999)
This is a second untouched period that lies *before* the in-sample window. It was never used for selection. Total-return data exist from 1988, so buy-and-hold is credited with dividends here, just as in the main study.

In [ ]:
PRE = (CONFIG["presample_start"], CONFIG["presample_end"])
pre_ok = spx.close.index[0] <= pd.Timestamp(PRE[0]) - pd.DateOffset(days=400)
PRESAMPLE = None
if not pre_ok:
    print("Not enough S&P 500 history for the pre-sample holdout; skipped.")
else:
    pre_frames = {k: rm.window(f, *PRE) for k, f in spx_frames.items()}
    pre_cash = rm.window(spx.cash, *PRE)
    pre_series = {**{k: f["net"] for k, f in pre_frames.items()}, "S&P 500 B&H": rm.window(spx.ret, *PRE)}
    PRESAMPLE = rm.stats_table(pre_series, pre_cash, pre_frames)
    show(PRESAMPLE.loc[["CAGR", "Ann. volatility", "Sharpe", "Max drawdown", "Calmar", "Time in market", "Switches / year"]])
    fig, ax = plt.subplots(figsize=(11, 4.2))
    rm.plot_wealth(pre_series, f"Pre-sample holdout {PRE[0][:4]}-{PRE[1][:4]}: growth of $1", ax=ax, colors=PALETTE)
    plt.tight_layout()
    plt.show()

### 10.5 Regime check: has short-horizon momentum decayed?
Over 1960–1989, short lookbacks (especially 1W) beat buy-and-hold by a wide margin. The usual explanation is **positive short-horizon autocorrelation** in index returns: up days and weeks tended to follow up days and weeks (Lo & MacKinlay, 1988). Part of this was an artifact of thinly traded stocks whose stale prices made the published index lag. The effect was arbitraged away as index futures (1982), index ETFs (1993), decimalisation (2001) and electronic trading arrived.

This exhibit measures that directly on the data: the lag-1 autocorrelation of daily and weekly S&P 500 returns by decade, and how much each lookback's Sharpe beat or trailed buy-and-hold in each decade. It is **descriptive and post-hoc**: it plays no part in selection.

Caveat: before 1988, returns are price-only (no total-return data). Buy-and-hold therefore misses dividends of 3–5% a year while the strategy earns T-bills in cash, which flatters timing in the early decades.

In [ ]:
px_ret = spx.close.pct_change()
wk_ret = spx.close.resample("W-FRI").last().pct_change()
bh_ex = spx.ret - spx.cash
rows = {}
first_dec = max(1960, (spx.close.index[0] + pd.DateOffset(years=1)).year // 10 * 10)
for dec in range(first_dec, OOS_END.year + 1, 10):
    a, b = f"{dec}-01-01", f"{dec + 9}-12-31"
    d, w = px_ret.loc[a:b].dropna(), wk_ret.loc[a:b].dropna()
    if len(d) < 200 or rm.window(spx_frames["1Y"]["signal"], a, b).isna().any():
        continue
    row = {"Daily autocorr (lag 1)": d.autocorr(1), "Weekly autocorr (lag 1)": w.autocorr(1)}
    bh_sr = rm.sharpe(rm.window(bh_ex, a, b))
    for k, f in spx_frames.items():
        row[f"{k} Sharpe minus B&H Sharpe"] = rm.sharpe(rm.window(f["net"] - f["cash"], a, b)) - bh_sr
    rows[f"{dec}s" + (" (partial)" if pd.Timestamp(b) > OOS_END else "")] = row
regime = pd.DataFrame.from_dict(rows, orient="index")
print("S&P 500 return autocorrelation and lookback edge vs buy & hold, by decade")
show(regime, by="columns")

ROLL5 = 5 * rm.TRADING_DAYS
start5 = regime.index[0][:4] + "-01-01" if len(regime) else CONFIG["is_start"]
fig, ax = plt.subplots(figsize=(11, 4.4))
bh_roll = rm.rolling_sharpe(rm.window(bh_ex, start5, OOS_END), ROLL5)
for k, f in spx_frames.items():
    diff = rm.rolling_sharpe(rm.window(f["net"] - f["cash"], start5, OOS_END), ROLL5) - bh_roll
    ax.plot(diff.index, diff.to_numpy(), color=PALETTE[k], lw=1.3, label=k, zorder=3)
ax.axhline(0, color=rm.INK["axis"], lw=1)
for a_, b_, lab in ((CONFIG["presample_start"], CONFIG["presample_end"], "holdout"),
                    (CONFIG["is_start"], CONFIG["is_end"], "in-sample"),
                    (CONFIG["oos_start"], OOS_END, "OOS")):
    ax.axvspan(pd.Timestamp(a_), pd.Timestamp(b_), color="#f0efec" if lab != "in-sample" else "#e8f0fb", zorder=0, lw=0)
    ax.text(pd.Timestamp(a_), 1.0, f" {lab}", transform=ax.get_xaxis_transform(), va="top", fontsize=8,
            color=rm.INK["secondary"])
ax.set_title("Rolling 5-year Sharpe ratio minus buy & hold's, S&P 500 (above 0 = lookback beat buy & hold)")
ax.set_ylabel("Sharpe difference")
ax.legend(loc="lower left", ncol=4)
plt.tight_layout()
plt.show()

In [ ]:
def by_decade(m, frame):
    rows = {}
    full = rm.window(frame, CONFIG["is_start"], OOS_END)
    for dec in range(int(CONFIG["is_start"][:4]) // 10 * 10, OOS_END.year + 1, 10):
        a, b = f"{dec}-01-01", f"{dec + 9}-12-31"
        w = rm.window(full, a, b)
        if len(w) < 200:
            continue
        bh = rm.window(m.ret, a, b)
        cash = rm.window(m.cash, a, b)
        label = f"{dec}s" + (" (partial)" if pd.Timestamp(b) > OOS_END else "")
        rows[label] = {
            f"{SELECTED} Sharpe": rm.sharpe(w["net"] - w["cash"]), "B&H Sharpe": rm.sharpe(bh - cash),
            f"{SELECTED} CAGR": rm.perf_stats(w["net"], cash)["CAGR"], "B&H CAGR": rm.perf_stats(bh, cash)["CAGR"],
            f"{SELECTED} max DD": rm.drawdown(w["net"]).min(), "B&H max DD": rm.drawdown(bh).min(),
        }
    return pd.DataFrame(rows).T


print("10.6 Sub-period stability by decade")
show(pd.concat({"S&P 500": by_decade(spx, spx_frames[SELECTED]),
                "Nasdaq-100": by_decade(ndx, ndx_frames[SELECTED])}), by="columns")

## 11. Conclusions

The scorecard and verdict below are generated from the numbers computed above, using rules fixed in §1, so they stay correct whenever the data are refreshed.

In [ ]:
s_oos, b_oos = oos_table[LBL], oos_table["S&P 500 B&H"]
_w = rm.window(spx_frames_2x[SELECTED], *OOS)
sr_2x_oos = rm.sharpe(_w["net"] - _w["cash"])
_w = rm.window(rm.backtest(spx_frames[SELECTED]["signal"], spx.ret, spx.cash, COST, LAG + 1), *OOS)
sr_delay_oos = rm.sharpe(_w["net"] - _w["cash"])
n_oos, nb_oos = nq_oos_table[NQ_LBL], nq_oos_table["Nasdaq-100 B&H"]
dsr4 = dsr.loc["DSR", "Candidate set (N=4)"]
pre_pass = None
if PRESAMPLE is not None:
    pre_pass = bool(PRESAMPLE.loc["Sharpe", SELECTED] >= PRESAMPLE.loc["Sharpe", "S&P 500 B&H"])

checks = {
    "1. Selected lookback survived all in-sample gates": bool(SELECTED_SURVIVED),
    "2a. OOS Sharpe >= S&P 500 buy & hold": bool(s_oos["Sharpe"] >= b_oos["Sharpe"]),
    "2b. OOS Sharpe gain is significant (bootstrap p < 0.05)": bool(boot["p-value (H0: no improvement)"] < 0.05),
    "3. OOS max drawdown shallower than buy & hold": bool(s_oos["Max drawdown"] > b_oos["Max drawdown"]),
    "4a. Low overfitting risk: PBO < 0.5 (candidate set)": bool(pbo_cand["pbo"] < 0.5),
    "4b. Survives deflation: DSR > 0.95 (N = 4)": bool(dsr4 > 0.95),
    "5. Timing skill beyond exposure (circular-shift p < 0.05)": bool(shift["p-value"] < 0.05),
    "6. Transfers to NQ: OOS Sharpe >= Nasdaq-100 buy & hold": bool(n_oos["Sharpe"] >= nb_oos["Sharpe"]),
    "7. Beats B&H Sharpe at 2x cost and with a 1-day delay": bool(
        sr_2x_oos >= bh_oos_sr and sr_delay_oos >= bh_oos_sr),
    f"Extra: holds in the {CONFIG['presample_start'][:4]}-{CONFIG['presample_end'][2:4]} pre-sample holdout": pre_pass,
}
score = pd.Series({k: ("PASS" if v else "FAIL") if v is not None else "n/a" for k, v in checks.items()},
                  name="result")
display(score.to_frame())

lo, hi = boot["95% CI low"], boot["95% CI high"]
oos_switches = int(round(sel_oos["turnover"].sum()))
if oos_switches == 0 or s_oos["Time in market"] > 0.995:
    verdict = (f"**INCONCLUSIVE OUT OF SAMPLE.** The {SELECTED} rule stayed invested for the whole out-of-sample "
               f"window ({oos_switches} switches), so its OOS return is identical to buy-and-hold by construction "
               "and the OOS period carries no information about timing skill. Judge the rule on the in-sample "
               "evidence (section 6-7) and the Nasdaq-100 full-history transfer (section 9), and extend the "
               "forward test as new data arrive.")
else:
    better = checks["2a. OOS Sharpe >= S&P 500 buy & hold"]
    significant = checks["2b. OOS Sharpe gain is significant (bootstrap p < 0.05)"]
    robust = {"transfers to the Nasdaq-100": checks["6. Transfers to NQ: OOS Sharpe >= Nasdaq-100 buy & hold"],
              "PBO < 0.5": checks["4a. Low overfitting risk: PBO < 0.5 (candidate set)"]}
    gap = (f"Sharpe {s_oos['Sharpe']:.2f} vs {b_oos['Sharpe']:.2f}, difference {boot['SR difference']:+.2f}, "
           f"95% CI [{lo:+.2f}, {hi:+.2f}]")
    if better and significant and all(robust.values()):
        verdict = ("**SUPPORTED.** The frozen lookback beat buy-and-hold on a risk-adjusted basis out of sample "
                   f"({gap}), the improvement is statistically significant, it carried over to the Nasdaq-100, "
                   "and selection-overfitting risk is low.")
    elif better and significant:
        failed = ", ".join(k for k, v in robust.items() if not v)
        verdict = ("**PROMISING BUT FRAGILE.** Out of sample the strategy beat S&P 500 buy-and-hold with a "
                   f"statistically significant Sharpe improvement ({gap}), but it fails: {failed}.")
    elif hi < 0:
        verdict = ("**REJECTED.** Out of sample the strategy delivered a *significantly lower* Sharpe ratio than "
                   f"buy-and-hold ({gap}).")
    elif checks["3. OOS max drawdown shallower than buy & hold"] and s_oos["Calmar"] >= b_oos["Calmar"]:
        verdict = ("**A RISK OVERLAY, NOT AN ALPHA SOURCE.** Out of sample the strategy's Sharpe ratio is "
                   f"statistically indistinguishable from buy-and-hold ({gap}), while its maximum drawdown was "
                   f"{s_oos['Max drawdown']:.1%} vs {b_oos['Max drawdown']:.1%} with a better Calmar ratio. "
                   "Its value is in drawdown control and tail protection, paid for with lower participation in rallies.")
    elif better:
        verdict = ("**MIXED.** The strategy's out-of-sample Sharpe was higher than buy-and-hold's, but not "
                   f"significantly so ({gap}), and it did not deliver better return per unit of drawdown.")
    else:
        verdict = ("**NOT SUPPORTED.** Out of sample the strategy trailed buy-and-hold on a risk-adjusted basis "
                   f"({gap}; Calmar {s_oos['Calmar']:.2f} vs {b_oos['Calmar']:.2f}), and any drawdown reduction "
                   "did not pay for the return given up. Over this window the gap is not statistically significant.")

wf_sr = wf_tbl.loc["Sharpe", "Walk-forward"]
text = f'''
### Verdict
{verdict}

### Key findings
* **Selection (in-sample {IS_TXT}).** {SELECTED} was selected. It {status}. In-sample Sharpe was {gates.loc[SELECTED, 'IS Sharpe']:.2f} vs {bh_is['Sharpe']:.2f} for buy-and-hold; max drawdown {gates.loc[SELECTED, 'IS max DD']:.1%} vs {bh_is['Max drawdown']:.1%}.
* **Lookahead audit.** {"All checks passed" if (AUDIT['result'] == 'PASS').all() else "At least one check FAILED; see section 5"}. The deliberately leaky lag-0 variant was detected by the same-bar test and shows how a one-day error would inflate results.
* **Overfitting.** PBO = {pbo_cand['pbo']:.0%} on the candidate set and {pbo_grid['pbo']:.0%} on the 23-lookback grid. DSR = {dsr4:.1%} (N = 4) and {dsr.iloc[-1, 1]:.1%} (N = {len(GRID_DAYS)}). The IS-vs-OOS Sharpe rank correlation across the grid was {stability.iloc[-1]:+.2f}. Walk-forward re-selection earned an OOS Sharpe of {wf_sr:.2f} vs {s_oos['Sharpe']:.2f} for the frozen choice.
* **Out-of-sample vs S&P 500 ({OOS_TXT}).** CAGR {s_oos['CAGR']:.1%} vs {b_oos['CAGR']:.1%}; Sharpe {s_oos['Sharpe']:.2f} vs {b_oos['Sharpe']:.2f}; max drawdown {s_oos['Max drawdown']:.1%} vs {b_oos['Max drawdown']:.1%}; time in market {s_oos['Time in market']:.0%}; {s_oos['Switches / year']:.1f} switches a year. CAPM beta {ab['Beta']:.2f}, alpha {ab['Alpha (ann.)']:+.1%} a year (HAC t = {ab['Alpha t-stat (HAC)']:.2f}). Circular-shift timing p-value: {shift['p-value']:.3f}.
* **Nasdaq-100 transfer (no re-fitting).** OOS Sharpe {n_oos['Sharpe']:.2f} vs {nb_oos['Sharpe']:.2f} for NQ buy-and-hold; max drawdown {n_oos['Max drawdown']:.1%} vs {nb_oos['Max drawdown']:.1%}. Run independently, the Nasdaq-100 in-sample gates would choose **{NQ_OWN}**.
* **Robustness.** Break-even cost vs buy-and-hold: {fmt_be(be[SELECTED]) + (" bp per switch" if isinstance(be[SELECTED], float) and np.isfinite(be[SELECTED]) else "")}. At 2x costs the OOS Sharpe is {sr_2x_oos:.2f}; with one extra day of execution delay it is {sr_delay_oos:.2f} (base {s_oos['Sharpe']:.2f}, buy-and-hold {b_oos['Sharpe']:.2f}).

### Caveats
* The out-of-sample window is {len(sel_oos)} trading days (~{len(sel_oos) / 252:.1f} years). The standard error of an annualised Sharpe over that span is about ±{1 / np.sqrt(len(sel_oos) / 252):.2f}, so differences smaller than that are noise.
* This is a single-asset test. Momentum timing on one index has low statistical power: about one independent "bet" per trend regime. Confidence intervals are wide by nature.
* Index levels are not directly tradable. ES/NQ futures or SPY/QQQ track them closely but add roll costs or fund fees (≈0.03–0.20% a year) and taxes on switches, which are ignored here.
* US large-cap indices are, with hindsight, among the best-performing equity markets of the past century. Evidence across many markets (Hurst, Ooi & Pedersen 2017) is the stronger basis for trend-following.
'''
display(Markdown(text))

## Appendix: methodology notes, limitations and references

**Return and risk definitions.** Daily simple returns. CAGR from compounded wealth. Volatility = daily standard deviation × √252. Sharpe = mean daily excess return over T-bills ÷ its standard deviation × √252. Sortino uses downside deviation of excess returns. Max drawdown is measured from the running peak, including starting capital.

**Probabilistic & Deflated Sharpe.** $\widehat{PSR}(SR^*) = \Phi\!\left(\frac{(\widehat{SR}-SR^*)\sqrt{T-1}}{\sqrt{1-\hat\gamma_3\widehat{SR}+\frac{\hat\gamma_4-1}{4}\widehat{SR}^2}}\right)$ with daily (non-annualised) Sharpe, skewness $\gamma_3$ and kurtosis $\gamma_4$. The DSR sets $SR^* = \sqrt{V[\widehat{SR}_n]}\left((1-\gamma)\Phi^{-1}(1-\tfrac1N)+\gamma\Phi^{-1}(1-\tfrac1{Ne})\right)$, the expected maximum Sharpe of $N$ skill-less trials, where $\gamma$ is the Euler–Mascheroni constant.

**PBO / CSCV.** The IS excess-return matrix ($T \times N$) is split into $S=16$ contiguous blocks. For each of the $\binom{16}{8}$ splits, the configuration with the best Sharpe on the "train" half is ranked on the "test" half, giving $\lambda = \text{logit}(\text{rank}/(N+1))$. PBO = $P(\lambda \le 0)$.

**Stationary bootstrap.** Politis & Romano (1994) resampling with geometric block lengths (mean 63 trading days), applied to paired daily excess returns so cross-correlation is preserved. The one-sided p-value uses the re-centred bootstrap distribution.

**Known limitations.**
* Yahoo Finance index data are generally reliable but not audited. The data-quality table flags gaps and extreme moves.
* `^XNDX` (Nasdaq-100 total return) may be unavailable or short on Yahoo. Where missing, Nasdaq P&L uses price returns, which understates buy-and-hold by the index's ~1% dividend yield.
* Pre-1988 S&P returns are price-only. This affects only the regime exhibit in §10.5.
* The four candidate lookbacks were specified a priori. The 23-point grid is purely diagnostic and its DSR (N = 23) is the conservative bound.

**References**
* Antonacci, G. (2014). *Dual Momentum Investing.* McGraw-Hill.
* Bailey, D., Borwein, J., López de Prado, M. & Zhu, Q. (2017). The Probability of Backtest Overfitting. *Journal of Computational Finance* 20(4).
* Bailey, D. & López de Prado, M. (2012). The Sharpe Ratio Efficient Frontier. *Journal of Risk* 15(2).
* Bailey, D. & López de Prado, M. (2014). The Deflated Sharpe Ratio. *Journal of Portfolio Management* 40(5).
* Faber, M. (2007). A Quantitative Approach to Tactical Asset Allocation. *Journal of Wealth Management* 9(4).
* Harvey, C., Liu, Y. & Zhu, H. (2016). … and the Cross-Section of Expected Returns. *Review of Financial Studies* 29(1).
* Hurst, B., Ooi, Y. H. & Pedersen, L. H. (2017). A Century of Evidence on Trend-Following Investing. *Journal of Portfolio Management* 44(1).
* Lo, A. (2002). The Statistics of Sharpe Ratios. *Financial Analysts Journal* 58(4).
* Mertens, E. (2002). Comments on Variance of the IID Estimator in Lo (2002). Working paper.
* Moskowitz, T., Ooi, Y. H. & Pedersen, L. H. (2012). Time Series Momentum. *Journal of Financial Economics* 104(2).
* Politis, D. & Romano, J. (1994). The Stationary Bootstrap. *JASA* 89(428).